# Drug Discovery GRPO — Kaggle Runbook

End-to-end on a Kaggle GPU notebook (T4 x1 / x2 or P100):

1. Clone the repo from GitHub
2. Install all dependencies (TRL, Unsloth 4-bit, RDKit, FastAPI, etc.)
3. Build the disease/target dataset (`prepare_dataset.py`)
4. Boot the FastAPI env server inside the notebook
5. Run multi-disease GRPO training (`train.py`)
6. Evaluate on the held-out test split (`evaluate.py`)
7. Run inference on a brand-new (unseen) disease (`infer.py`)

> Before running: in **Kaggle Notebook → Settings**, enable **Internet** and pick an
> **Accelerator**. Then run the cells in order.
>
> ⚠️ **You MUST pick `GPU T4 x2`, NOT `GPU P100`.** The P100 (CUDA capability
> 6.0) is no longer supported by recent PyTorch wheels (`sm_70+` only) — training
> will silently fall back to CPU and take forever. The T4 (`sm_75`) works fine.

## 0. Choose run knobs

These are the only values you should typically change. They are exported as env
vars and consumed by the CLI flags below. Defaults are tuned for a Kaggle T4.

In [ ]:
import os

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/VasuBB/drug-discovery-sim-env.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'main')
WORK_DIR = '/kaggle/working/drug-discovery-sim-env' if os.path.isdir('/kaggle') else os.path.abspath('drug-discovery-sim-env')

# Dataset prep
# Goal: ~90 train + ~10 eval diseases.
# prepare_dataset.py asks Open Targets for NUM_DISEASES rows, then drops any
# whose top-target druggability < MIN_DRUGGABILITY (~25-30% are dropped). With
# NUM_DISEASES=100 you typically land near 70-85 kept rows; bump to 140 if you
# want to hit ~100 kept (~90 train + ~10 test under test_fraction=0.10).
NUM_DISEASES = int(os.environ.get('NUM_DISEASES', '100'))    # bump to 5000+ for the full dataset
TEST_FRACTION = float(os.environ.get('TEST_FRACTION', '0.10'))
MIN_DRUGGABILITY = float(os.environ.get('MIN_DRUGGABILITY', '0.30'))
KNOWN_DRUGS_PER_TARGET = int(os.environ.get('KNOWN_DRUGS_PER_TARGET', '8'))

# Training
MODEL_NAME = os.environ.get('MODEL_NAME', 'Qwen/Qwen2.5-0.5B-Instruct')
NUM_TRAIN_STEPS = int(os.environ.get('NUM_TRAIN_STEPS', '50'))
GROUP_SIZE = int(os.environ.get('GROUP_SIZE', '4'))
ENV_PORT = int(os.environ.get('ENV_PORT', '8000'))
# Env transport: set USE_HF=1 to talk to the public HF Space instead of
# booting a local uvicorn (skips cell #12). Useful for judges who want
# to verify the deployed env end-to-end without provisioning compute.
USE_HF = os.environ.get('USE_HF', '0') == '1'
HF_BASE_URL = os.environ.get('HF_BASE_URL', 'https://vasuboda-drug-discovery-sim-env.hf.space')
BASE_URL = HF_BASE_URL if USE_HF else f'http://127.0.0.1:{ENV_PORT}'

# Evaluation / inference
EVAL_LIMIT = int(os.environ.get('EVAL_LIMIT', '20'))          # cap test diseases (None = use all)
INFER_DISEASE = os.environ.get('INFER_DISEASE', 'Idiopathic pulmonary fibrosis')

print('Work dir:', WORK_DIR)
print('Model:', MODEL_NAME, '| GRPO steps:', NUM_TRAIN_STEPS, '| group size:', GROUP_SIZE)
print('Env transport:', 'REMOTE HF Space' if USE_HF else 'LOCAL uvicorn', '->', BASE_URL)

## 1. Clone the repo

In [ ]:
import os, subprocess, sys

if not os.path.isdir(WORK_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, WORK_DIR])
else:
    subprocess.check_call(['git', '-C', WORK_DIR, 'fetch', '--depth', '1', 'origin', REPO_BRANCH])
    subprocess.check_call(['git', '-C', WORK_DIR, 'checkout', REPO_BRANCH])
    subprocess.check_call(['git', '-C', WORK_DIR, 'reset', '--hard', f'origin/{REPO_BRANCH}'])

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print('Working in', os.getcwd())

## 2. Install dependencies

Unsloth ships its own torch/triton wheels, so we install it first and let pip
resolve everything else around it. RDKit and TRL come from the `[chem]` and
`[training]` extras.

In [ ]:
# Why this cell is structured the way it is:
#
# - TRL 0.20+ hard-imports `mergekit` in `trl/trainer/callbacks.py`. Newer
#   mergekit declares `Dict[str, torch.Tensor]` fields that require
#   pydantic v1 / pydantic <2.11 (newer pydantic refuses arbitrary types
#   without explicit `ConfigDict(arbitrary_types_allowed=True)`).
#   On Kaggle this manifests as either:
#     * `ModuleNotFoundError: No module named 'mergekit'`, or
#     * `PydanticSchemaGenerationError: <class 'torch.Tensor'>`.
#
# - Both go away if we pin TRL to the 0.18.x series, which still has
#   `GRPOTrainer` (satisfies the hackathon "trl >= 0.18" requirement) but
#   does NOT auto-import mergekit. As a belt-and-suspenders we also force
#   pydantic to <2.11 at the end.
#
# - OpenEnv ships on PyPI as `openenv-core` (import path `openenv.core`).
#   Our `openenv_compat.py` falls back to a local shim if missing, but we
#   install it explicitly to satisfy the hackathon "OpenEnv (latest release)"
#   requirement.
#
# - We deliberately drop `%%capture` so silent install failures (e.g. a
#   non-existent version pin) are visible in the cell output.

!pip install --upgrade pip --quiet

# 1) OpenEnv (mandatory).
!pip install --quiet openenv-core || pip install --quiet openenv || echo '[warn] openenv not on PyPI; using bundled shim in drug_discovery_env/openenv_compat.py'

# 2) RL stack. CRITICAL: Kaggle ships TRL 0.24 + a broken dev `transformers`
#    system-wide. Plain `pip install ...>=X` is a no-op when newer is already
#    installed. We must **uninstall + reinstall** specific compatible versions.

# 2a) Wipe Kaggle's broken dev versions of the whole RL/transformers stack.
!pip uninstall -y trl transformers tokenizers huggingface-hub 2>/dev/null || true

# 2b) Install transformers 4.52.4 WITH its deps. This pulls the matching
#     tokenizers (0.21.x) and huggingface_hub (0.30.x) automatically and
#     is the version TRL 0.18.2 declares it needs (>=4.50.0).
#     transformers 4.52.4 is verified to have `is_rich_available`,
#     `is_flash_attn_2_available`, and is internally consistent (no
#     dangling `is_flash_attn_4_available` reference).
!pip install --no-cache-dir 'transformers==4.52.4'

# 2c) TRL 0.18.2 -- --no-deps so it doesn't try to upgrade transformers.
!pip install --no-cache-dir --no-deps 'trl==0.18.2'

# 2d) accelerate >=1.3.0 (transformers 4.52 calls `unwrap_model(keep_torch_compile=...)`
#     which was added in accelerate 1.3.0). Use 1.7.0 -- known stable.
!pip uninstall -y accelerate 2>/dev/null || true
!pip install --no-cache-dir --no-deps 'accelerate==1.7.0'

# 2e) Companion deps -- these are usually fine on Kaggle, install only if missing.
!pip install --quiet 'datasets>=2.20' 'peft>=0.11,<0.14' 'bitsandbytes>=0.43'

# Sanity: confirm versions actually changed.
import importlib, sys
for n in list(sys.modules):
    if (n == 'trl' or n.startswith('trl.') \
        or n == 'transformers' or n.startswith('transformers.') \
        or n == 'accelerate' or n.startswith('accelerate.') \
        or n == 'tokenizers' or n.startswith('tokenizers.') \
        or n == 'huggingface_hub' or n.startswith('huggingface_hub.')):
        del sys.modules[n]
import trl, transformers, accelerate, tokenizers, huggingface_hub
print(f'[install] trl              = {trl.__version__}')
print(f'[install] transformers     = {transformers.__version__}')
print(f'[install] accelerate       = {accelerate.__version__}')
print(f'[install] tokenizers       = {tokenizers.__version__}')
print(f'[install] huggingface_hub  = {huggingface_hub.__version__}')
assert trl.__version__.startswith('0.18'), \
    f'TRL downgrade FAILED (got {trl.__version__}).'
assert transformers.__version__.startswith('4.52'), \
    f'transformers pin FAILED (got {transformers.__version__}).'
assert huggingface_hub.__version__.startswith('0.'), \
    f'huggingface_hub pin FAILED (got {huggingface_hub.__version__}, need <1.0).'

# 3) Cheminformatics + retrieval.
!pip install --quiet 'rdkit>=2024.3.1' rank-bm25 'sentence-transformers>=2.7'

# 4) Env server runtime.
!pip install --quiet 'fastapi>=0.115' 'uvicorn[standard]>=0.30' \
    'pydantic-settings>=2.2' 'PyYAML>=6.0' requests numpy networkx

# 5) Unsloth (optional, GPU-only).
!pip install --quiet 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' \
    || echo '[warn] Unsloth not installed; falling back to plain HF backend'

# 6) Repo install with --no-deps so it doesn't drag in unrelated `openenv`
#    from PyPI or upgrade our pinned versions.
!pip install --quiet --no-deps -e .

print('install cell finished')

In [ ]:
# Install a minimal `mergekit` STUB package so `from trl import GRPOTrainer`
# succeeds regardless of which TRL version Kaggle ships.
#
# Why this is needed: TRL 0.20+ does an UNCONDITIONAL
#   `from ..mergekit_utils import MergeConfig, merge_models, upload_model_to_hf`
# in `trl/trainer/callbacks.py`, and `mergekit_utils.py` then does
#   `from mergekit.config import MergeConfiguration`
#   `from mergekit.merge import MergeOptions, run_merge`
# at module top-level. If we install REAL mergekit it pulls in pydantic v1
# patterns that crash with pydantic >= 2.11 (PydanticSchemaGenerationError on
# torch.Tensor). If we DON'T install mergekit, the import above fails outright.
#
# GRPO training never calls into mergekit — it's only used for model merging.
# So a 4-line stub package satisfies the import without dragging in the bug.
#
# We write it directly into the system `dist-packages` so subprocess `python`
# calls (e.g. `!python -m drug_discovery_env.scripts.train`) see it too.

import sys, pathlib, site, importlib

site_dirs = [pathlib.Path(d) for d in site.getsitepackages()]
target = next((d for d in site_dirs if d.exists()), None)
if target is None:
    target = pathlib.Path('/usr/local/lib/python3.12/dist-packages')
print(f'[mergekit-stub] writing into {target}')

mk = target / 'mergekit'
mk.mkdir(parents=True, exist_ok=True)
(mk / '__init__.py').write_text('__version__ = "0.0.0-stub-for-trl"\n')
(mk / 'config.py').write_text(
    '"""Stub provided by drug-discovery-sim-env to satisfy trl imports."""\n'
    'class MergeConfiguration: pass\n'
)
(mk / 'merge.py').write_text(
    '"""Stub provided by drug-discovery-sim-env to satisfy trl imports."""\n'
    'class MergeOptions: pass\n'
    'def run_merge(*args, **kwargs):\n'
    '    raise NotImplementedError("mergekit stub: GRPO does not use model merging")\n'
)
(mk / 'card.py').write_text(
    'def generate_card(*args, **kwargs): return ""\n'
)

# A minimal dist-info so importlib.metadata.version("mergekit") returns
# something (some versions of trl probe via metadata, not just import).
dist_info = target / 'mergekit-0.0.0.dist-info'
dist_info.mkdir(exist_ok=True)
(dist_info / 'METADATA').write_text(
    'Metadata-Version: 2.1\nName: mergekit\nVersion: 0.0.0\n'
)
(dist_info / 'INSTALLER').write_text('drug-discovery-sim-env\n')
(dist_info / 'RECORD').write_text('')
(dist_info / 'WHEEL').write_text('Wheel-Version: 1.0\nGenerator: stub\n')

# --- llm_blender stub (TRL 0.24 callbacks.py auto-imports it) -----------
lblender = target / 'llm_blender'
lblender.mkdir(parents=True, exist_ok=True)
(lblender / '__init__.py').write_text(
    '__version__ = "0.0.0-stub-for-trl"\n'
    'class Blender:\n'
    '    def __init__(self, *a, **kw): pass\n'
    '    def loadranker(self, *a, **kw): pass\n'
    '    def rank(self, *a, **kw): return []\n'
)
lb_dist = target / 'llm_blender-0.0.0.dist-info'
lb_dist.mkdir(exist_ok=True)
(lb_dist / 'METADATA').write_text(
    'Metadata-Version: 2.1\nName: llm_blender\nVersion: 0.0.0\n'
)
(lb_dist / 'INSTALLER').write_text('drug-discovery-sim-env\n')
(lb_dist / 'RECORD').write_text('')
(lb_dist / 'WHEEL').write_text('Wheel-Version: 1.0\nGenerator: stub\n')
print(f'[llm_blender-stub] wrote {lblender}')

# --- NUKE vllm + vllm_ascend completely ----------------------------------
# Stubbing vllm doesn't work because TRL does `from vllm import LLM` etc.
# Cleanest fix: remove the package + its dist-info entirely so
# `is_vllm_available()` returns False and TRL skips the whole block.
# (We never use vllm for inference -- the trainer uses HF generate().)
import shutil
for pkg_name in ('vllm', 'vllm_ascend'):
    pkg_dir = target / pkg_name
    if pkg_dir.exists():
        shutil.rmtree(pkg_dir, ignore_errors=True)
    for d in target.glob(f'{pkg_name}-*.dist-info'):
        shutil.rmtree(d, ignore_errors=True)
    for d in target.glob(f'{pkg_name}-*.egg-info'):
        shutil.rmtree(d, ignore_errors=True)
    for f in target.glob(f'{pkg_name}.py'):
        f.unlink(missing_ok=True)
    print(f'[nuke] removed {pkg_name} from {target}')

# Drop any cached vllm/vllm_ascend modules so future imports actually fail.
import importlib
importlib.invalidate_caches()

# --- disk-level fallback: nuke leftover TRL/transformers if pip didn't take -
import subprocess
def _force_pin(pkg_prefix, dist_glob, want_version, pip_name=None):
    import importlib
    pip_name = pip_name or pkg_prefix
    for n in list(sys.modules):
        if n == pkg_prefix or n.startswith(pkg_prefix + '.'):
            del sys.modules[n]
    try:
        mod = importlib.import_module(pkg_prefix)
        if mod.__version__.startswith(want_version):
            return
        print(f'[stub] {pkg_prefix} still at {mod.__version__}, nuking from disk...')
    except Exception:
        print(f'[stub] {pkg_prefix} not importable, reinstalling...')
    for p in target.glob(dist_glob):
        shutil.rmtree(p, ignore_errors=True) if p.is_dir() else p.unlink(missing_ok=True)
    # Also nuke the dashed dist-info variant (huggingface-hub vs huggingface_hub).
    for p in target.glob(dist_glob.replace('_', '-')):
        shutil.rmtree(p, ignore_errors=True) if p.is_dir() else p.unlink(missing_ok=True)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           '--no-cache-dir', '--no-deps',
                           f'{pip_name}=={want_version}'])
    for n in list(sys.modules):
        if n == pkg_prefix or n.startswith(pkg_prefix + '.'):
            del sys.modules[n]
    mod = importlib.import_module(pkg_prefix)
    print(f'[stub] {pkg_prefix} now at {mod.__version__}')

_force_pin('trl',          'trl*',          '0.18.2')
_force_pin('transformers', 'transformers*', '4.52.4')

# Drop any cached trl/mergekit/llm_blender/vllm modules from THIS kernel so the
# next import picks up the freshly-written stubs.
for name in list(sys.modules):
    if (name == 'mergekit' or name.startswith('mergekit.') \
        or name == 'llm_blender' or name.startswith('llm_blender.') \
        or name == 'vllm' or name.startswith('vllm.') \
        or name == 'vllm_ascend' or name.startswith('vllm_ascend.') \
        or name == 'trl' or name.startswith('trl.') \
        or name == 'transformers' or name.startswith('transformers.') \
        or name == 'tokenizers' or name.startswith('tokenizers.') \
        or name == 'huggingface_hub' or name.startswith('huggingface_hub.') \
        or name == 'datasets' or name.startswith('datasets.') \
        or name == 'peft' or name.startswith('peft.')):
        del sys.modules[name]

# Verify.
import mergekit  # type: ignore
from mergekit.config import MergeConfiguration  # noqa: F401
from mergekit.merge import MergeOptions, run_merge  # noqa: F401
import llm_blender  # type: ignore
print(f'[mergekit-stub]    OK  ({mergekit.__file__}, version={mergekit.__version__})')
print(f'[llm_blender-stub] OK  ({llm_blender.__file__}, version={llm_blender.__version__})')

In [ ]:
import importlib, torch, warnings

mods = ['pydantic', 'pydantic_core', 'transformers', 'trl', 'mergekit',
        'llm_blender', 'datasets', 'peft', 'accelerate', 'rdkit', 'fastapi',
        'uvicorn', 'drug_discovery_env']
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f'  {m:24s} OK   {getattr(mod, "__version__", "")}')
    except Exception as e:
        print(f'  {m:24s} FAIL {e}')

# Note: pydantic version no longer matters because we install a stub mergekit
# (previous cell) that removes the pydantic-Tensor schema bug. Just print it.
import pydantic
print(f'  pydantic version          {pydantic.VERSION}')

# OpenEnv check -- the real package or our bundled shim must import.
try:
    from drug_discovery_env.openenv_compat import HTTPEnvServer  # noqa: F401
    print('  openenv (or shim)        OK')
except Exception as e:
    print('  openenv                  FAIL', e)

# CRITICAL: import GRPOTrainer here so a broken stack blows up NOW (1 second)
# rather than after the 5-minute dataset build.
try:
    from trl import GRPOConfig, GRPOTrainer  # noqa: F401
    import trl
    print(f'  trl.GRPOTrainer          OK   (trl {trl.__version__})')
except Exception as e:
    raise RuntimeError(
        f'trl.GRPOTrainer failed to import: {e}\n\n'
        'Fix: re-run the install cell (it force-reinstalls trl==0.18.2) and '
        'the stub cell (mergekit + llm_blender), then **Restart Kernel** and '
        'run all. If a NEW optional dep is named in the error, add a stub for '
        'it in the stub cell using the same pattern as mergekit/llm_blender.'
    ) from e

# GPU sanity -- abort early if we got a P100 (cuda capability 6.0).
print()
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU: {name}  (compute capability {cap[0]}.{cap[1]})')
    if cap[0] < 7:
        warnings.warn(
            f'\n*** {name} (sm_{cap[0]}{cap[1]}) is NOT supported by the '
            f'installed PyTorch wheel (needs sm_70+). Training will fall back '
            f'to CPU and be unusably slow. Switch the Kaggle accelerator to '
            f'"GPU T4 x2" and re-run the notebook.',
            stacklevel=2,
        )
else:
    print('[warn] no CUDA -- training on CPU will be very slow.')

## 3. Build the disease / target / known-drugs dataset

One-time fetch from Open Targets + ChEMBL. Set `NUM_DISEASES` higher (e.g. 5500)
for a full run. Internet must be enabled.

In [ ]:
!python -m drug_discovery_env.scripts.prepare_dataset \
    --num-diseases {NUM_DISEASES} \
    --test-fraction {TEST_FRACTION} \
    --min-druggability {MIN_DRUGGABILITY} \
    --known-drugs-per-target {KNOWN_DRUGS_PER_TARGET}

import json, pathlib
manifest = json.loads(pathlib.Path('data/diseases.manifest.json').read_text())
manifest

## 4. Boot the env transport

By default we launch uvicorn in a subprocess and wait for `/health` to
return 200 before moving on. Logs stream to `outputs/server.log`.

Set `USE_HF=1` (env var or in cell #2) to skip the local boot and use the
public HF Space at <https://vasuboda-drug-discovery-sim-env.hf.space> instead.


In [ ]:
import os, subprocess, time, requests

if USE_HF:
    # Remote HF Space -- no local server to boot. Just verify it answers.
    server_proc = None
    r = requests.get(f'{BASE_URL}/health', timeout=10.0)
    r.raise_for_status()
    print('remote HF env up:', BASE_URL, '->', r.json())
else:
    os.makedirs('outputs', exist_ok=True)
    log_handle = open('outputs/server.log', 'w', buffering=1)
    server_proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'drug_discovery_env.server.app:app',
         '--host', '127.0.0.1', '--port', str(ENV_PORT), '--log-level', 'warning'],
        stdout=log_handle, stderr=subprocess.STDOUT,
    )

    for attempt in range(60):
        try:
            r = requests.get(f'{BASE_URL}/health', timeout=2.0)
            if r.status_code == 200:
                print('local env server up:', r.json())
                break
        except Exception:
            pass
        time.sleep(1.0)
    else:
        raise RuntimeError('env server failed to start; check outputs/server.log')


## 5. GRPO training

Multi-disease live-rollout training. Per-turn JSONL traces (tool calls,
literature queries, reasoning, reward breakdown) are written under
`outputs/grpo/logs/<run-id>/`.

In [ ]:
!mkdir -p outputs/grpo/logs
!PYTHONUNBUFFERED=1 python -u -m drug_discovery_env.scripts.train \
    --base-url {BASE_URL} \
    --model {MODEL_NAME} \
    --num-train-steps {NUM_TRAIN_STEPS} \
    --group-size {GROUP_SIZE} \
    --output-dir outputs/grpo \
    --log-dir outputs/grpo/logs \
    --run-id kaggle-run 2>&1 | tee outputs/grpo/train.log

## 5b. Training evidence — loss and reward plots

Reads the artifacts the trainer just wrote and renders:

- **Training loss** per optimizer step from `outputs/grpo/trainer_state.json`
  (HF/TRL `log_history`, populated because `logging_steps=1`).
- **GRPO mean reward / reward std** per step from the same `log_history`.
- **Per-episode terminal & total reward** from
  `outputs/grpo/logs/kaggle-run/runs.csv` with a rolling mean overlay.

PNGs are saved under `outputs/plots/` so you can attach them as proof of a
real training run.

In [ ]:
import json, csv, pathlib
import matplotlib.pyplot as plt

PLOT_DIR = pathlib.Path('outputs/plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# --- 1) HF/TRL trainer_state.json: loss + GRPO reward per optimizer step ---
state_path = None
for cand in [
    pathlib.Path('outputs/grpo/trainer_state.json'),
    *sorted(pathlib.Path('outputs/grpo').glob('checkpoint-*/trainer_state.json')),
]:
    if cand.exists():
        state_path = cand
        break

if state_path is None:
    print('[warn] no trainer_state.json found under outputs/grpo/ - did training finish?')
else:
    history = json.loads(state_path.read_text()).get('log_history', [])
    steps_loss, losses = [], []
    steps_reward, rewards, reward_stds = [], [], []
    for entry in history:
        step = entry.get('step')
        if step is None:
            continue
        if 'loss' in entry:
            steps_loss.append(step); losses.append(entry['loss'])
        if 'reward' in entry:
            steps_reward.append(step); rewards.append(entry['reward'])
            reward_stds.append(entry.get('reward_std', 0.0))

    print(f'[plot] {state_path}: {len(losses)} loss points, {len(rewards)} reward points')

    if losses:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(steps_loss, losses, linewidth=1.5, color='#c0392b')
        ax.set_xlabel('optimizer step'); ax.set_ylabel('loss')
        ax.set_title('GRPO training loss')
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(PLOT_DIR / 'training_loss.png', dpi=150)
        plt.show()

    if rewards:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(steps_reward, rewards, linewidth=1.5, color='#2c7fb8', label='mean reward')
        if any(s > 0 for s in reward_stds):
            import numpy as np
            r = np.array(rewards); s = np.array(reward_stds)
            ax.fill_between(steps_reward, r - s, r + s, color='#2c7fb8', alpha=0.15, label='+/-1 std')
        ax.set_xlabel('optimizer step'); ax.set_ylabel('reward')
        ax.set_title('GRPO group reward per step')
        ax.legend(); ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(PLOT_DIR / 'training_reward.png', dpi=150)
        plt.show()

# --- 2) runs.csv: per-episode terminal & total reward with rolling mean ---
runs_csv = pathlib.Path('outputs/grpo/logs/kaggle-run/runs.csv')
if runs_csv.exists():
    eps, terminal, total = [], [], []
    with runs_csv.open('r', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        for i, row in enumerate(reader):
            eps.append(i + 1)
            terminal.append(float(row['terminal_reward']))
            total.append(float(row['total_reward']))
    print(f'[plot] {runs_csv}: {len(eps)} episodes')

    def _rolling(xs, k=10):
        if len(xs) < 2:
            return xs
        k = min(k, len(xs))
        out = []
        for i in range(len(xs)):
            lo = max(0, i - k + 1)
            window = xs[lo:i+1]
            out.append(sum(window) / len(window))
        return out

    if eps:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(eps, total, alpha=0.35, color='#16a085', label='total reward (raw)')
        ax.plot(eps, _rolling(total, 10), color='#16a085', linewidth=2, label='total reward (rolling-10)')
        ax.plot(eps, terminal, alpha=0.35, color='#e67e22', label='terminal reward (raw)')
        ax.plot(eps, _rolling(terminal, 10), color='#e67e22', linewidth=2, label='terminal reward (rolling-10)')
        ax.set_xlabel('episode'); ax.set_ylabel('reward')
        ax.set_title(f'Per-episode rewards across {len(eps)} training rollouts')
        ax.legend(loc='best'); ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(PLOT_DIR / 'episode_rewards.png', dpi=150)
        plt.show()
else:
    print(f'[warn] {runs_csv} not found - no per-episode reward log.')

print('Saved plots to:', sorted(p.name for p in PLOT_DIR.glob('*.png')))

In [ ]:
import pathlib
for p in sorted(pathlib.Path('outputs/grpo').glob('*'))[:20]:
    print(p)
print('---')
csv_path = pathlib.Path('outputs/grpo/logs/kaggle-run/runs.csv')
if csv_path.exists():
    print(csv_path.read_text()[:2000])

## 6. Evaluate on the held-out test split

Computes the full panel: env reward, ADMET pass, oversight violations, budget
remaining, mean reasoning depth, and ChEMBL Tanimoto-to-known-drugs (precision@1
vs. cached known compounds for each test disease's target).

In [ ]:
!mkdir -p outputs/eval
!PYTHONUNBUFFERED=1 python -u -m drug_discovery_env.scripts.evaluate \
    --base-url {BASE_URL} \
    --checkpoint outputs/grpo \
    --limit {EVAL_LIMIT} \
    --run-id kaggle-eval 2>&1 | tee outputs/eval/run.log

In [ ]:
import json, pathlib
report = json.loads(pathlib.Path('outputs/eval/report.json').read_text())
report

## 7. Inference on an unseen disease

Pass any disease string. If it isn't in the cache the env falls back to live
Open Targets for the target lookup, then drives a full campaign with the
trained checkpoint.

In [ ]:
!mkdir -p outputs/infer
!PYTHONUNBUFFERED=1 python -u -m drug_discovery_env.scripts.infer \
    --base-url {BASE_URL} \
    --checkpoint outputs/grpo \
    --disease "{INFER_DISEASE}" \
    --out-dir outputs/infer 2>&1 | tee outputs/infer/run.log

import json, pathlib, re
slug = re.sub(r'[^a-z0-9]+', '-', INFER_DISEASE.lower()).strip('-')
summary = json.loads(pathlib.Path(f'outputs/infer/{slug}.json').read_text())
summary

## 8. Shut down the env server

In [ ]:
try:
    server_proc.terminate()
    server_proc.wait(timeout=5)
    print('env server stopped (exit', server_proc.returncode, ')')
except Exception as exc:
    print('shutdown error:', exc)
    server_proc.kill()

## 9. Package and download all results

Bundles the dataset cache, training logs, plots, evaluation report, inference
trace, and trained checkpoint into a single zip under `/kaggle/working/`.

On Kaggle:
- The zip appears in the right-hand **Output** panel of the notebook (click the
  download icon).
- Anything saved under `/kaggle/working/` is also persisted as the notebook's
  output, so individual files (the three PNG plots, `report.json`, etc.) are
  downloadable on their own too.

On Colab: a `files.download(...)` call is issued so the browser downloads the
zip directly.

In [ ]:
import os, shutil, pathlib, datetime

ON_KAGGLE = os.path.isdir('/kaggle/working')
DEST_DIR = pathlib.Path('/kaggle/working') if ON_KAGGLE else pathlib.Path.cwd()
DEST_DIR.mkdir(parents=True, exist_ok=True)

stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
bundle_name = f'drug_discovery_results_{stamp}'
staging = pathlib.Path('outputs') / '_bundle' / bundle_name
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True, exist_ok=True)

def _copy(src, dst_rel):
    src = pathlib.Path(src)
    if not src.exists():
        print(f'  [skip] {src} (missing)')
        return
    dst = staging / dst_rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
    print(f'  [ok]   {src} -> {dst.relative_to(staging)}')

print('Staging artefacts...')
_copy('data/diseases.jsonl',           'data/diseases.jsonl')
_copy('data/diseases.manifest.json',   'data/diseases.manifest.json')
_copy('outputs/plots',                 'plots')
_copy('outputs/eval/report.json',      'eval/report.json')
_copy('outputs/eval/per_disease.jsonl','eval/per_disease.jsonl')
_copy('outputs/infer',                 'infer')
_copy('outputs/grpo/logs',             'grpo/logs')
_copy('outputs/grpo/trainer_state.json','grpo/trainer_state.json')
_copy('outputs/grpo/all_results.json', 'grpo/all_results.json')
_copy('outputs/grpo/train_results.json','grpo/train_results.json')
_copy('outputs/server.log',            'server.log')
_copy('outputs/grpo/train.log',        'grpo/train.log')
_copy('outputs/eval/run.log',          'eval/run.log')
_copy('outputs/infer/run.log',         'infer/run.log')

# Copy the executed notebook itself so judges can see the run-with-outputs.
for nb in [pathlib.Path('/kaggle/working/__notebook_source__.ipynb'),
           pathlib.Path('notebooks/kaggle_drug_discovery_grpo.ipynb')]:
    if nb.exists():
        _copy(nb, 'notebook.ipynb')
        break

# Trained checkpoint (LoRA adapter is small; full HF checkpoint can be 1+ GB).
ckpt_dir = pathlib.Path('outputs/grpo')
if ckpt_dir.exists():
    INCLUDE_CHECKPOINT = bool(int(os.environ.get('INCLUDE_CHECKPOINT', '1')))
    if INCLUDE_CHECKPOINT:
        adapter_files = list(ckpt_dir.glob('adapter_*')) + list(ckpt_dir.glob('*.safetensors')) \
                      + list(ckpt_dir.glob('*.json')) + list(ckpt_dir.glob('tokenizer*'))
        for f in adapter_files:
            _copy(f, f'checkpoint/{f.name}')

zip_path = DEST_DIR / f'{bundle_name}.zip'
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=staging.parent, base_dir=bundle_name)
size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f'\nBundle ready: {zip_path}  ({size_mb:.1f} MB)')

if ON_KAGGLE:
    print('\nDownload on Kaggle:')
    print('  1. Open the right-hand "Output" panel of this notebook.')
    print(f'  2. Find {zip_path.name} and click the download icon.')
    print('  Individual files (plots/, eval/, etc.) are also under /kaggle/working/ in the same panel.')
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(zip_path)))
    except Exception:
        pass
else:
    try:
        from google.colab import files  # type: ignore
        files.download(str(zip_path))
    except Exception:
        try:
            from IPython.display import FileLink, display
            display(FileLink(str(zip_path)))
            print('Click the link above to download the bundle.')
        except Exception:
            print(f'Bundle saved at: {zip_path}')

## Outputs

Everything below is plain JSON / JSONL and can be downloaded from the Kaggle
notebook output panel:

- `data/diseases.jsonl` — cached disease/target/known-drug rows + `train`/`test` split
- `data/diseases.manifest.json` — row counts, sha256, fetch timestamp
- `outputs/grpo/` — trained checkpoint (HF format, or PEFT adapter)
- `outputs/grpo/logs/kaggle-run/*.jsonl` — per-episode turn-level traces
- `outputs/grpo/logs/kaggle-run/runs.csv` — one-line summary per training episode
- `outputs/plots/training_loss.png` — GRPO loss per optimizer step (training evidence)
- `outputs/plots/training_reward.png` — GRPO group reward (mean ± std) per step
- `outputs/plots/episode_rewards.png` — per-episode terminal & total reward with rolling mean
- `outputs/eval/report.json` — aggregate evaluation panel
- `outputs/eval/per_disease.jsonl` — per-test-disease metrics
- `outputs/infer/<slug>.json` — inference summary + reasoning trace